# 03 - Embedding Training Notebook

This notebook trains the GNN-based player embedding model.

## Contents

1. **Setup**: Load configs, initialize components
2. **Data Preparation**: Build event graphs and sequences
3. **Model Setup**: Initialize encoders and training
4. **Training Loop**: Train with contrastive loss
5. **Evaluation**: Validate and export embeddings

In [ ]:
# Imports
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import sys
sys.path.append('..')

# Project imports
from models.encoders import EventGraphEncoder, TemporalTransformerEncoder, FusionHead
from models.global_player_graph import GlobalPlayerGraph
from training import Trainer, TrainingConfig, InfoNCELoss
from training.metrics import EmbeddingMetrics, compute_recall_at_k
from configs import load_all_configs

print("Imports loaded successfully!")

## 1. Setup

In [ ]:
# Load configs
print("Loading configurations...")
try:
    configs = load_all_configs('../configs')
    model_config = configs['model']
    train_config = configs['train']
    print("✓ Configs loaded")
except:
    print("Using default configs")
    model_config = {}
    train_config = {}

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 2. Data Preparation

For demonstration, we create synthetic data. In production, use the actual data loading pipeline.

In [ ]:
class SyntheticPlayerDataset(Dataset):
    """Synthetic dataset for testing the training pipeline."""
    
    def __init__(self, n_players=100, n_samples_per_player=5, seq_length=50):
        self.n_players = n_players
        self.n_samples_per_player = n_samples_per_player
        self.seq_length = seq_length
        
        # Create synthetic data
        self.samples = []
        for player_id in range(n_players):
            for _ in range(n_samples_per_player):
                self.samples.append({
                    'player_id': player_id,
                    'event_embeddings': torch.randn(seq_length, 64),
                    'time_positions': torch.linspace(0, 90, seq_length),
                    'attention_mask': torch.ones(seq_length),
                    'n_events': seq_length + np.random.randint(-10, 10),
                })
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]


def collate_fn(batch):
    """Collate function for DataLoader."""
    return {
        'player_ids': torch.tensor([b['player_id'] for b in batch]),
        'sequences': {
            'event_embeddings': torch.stack([b['event_embeddings'] for b in batch]),
            'time_positions': torch.stack([b['time_positions'] for b in batch]),
            'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        },
        'events': {
            'node_features': torch.randn(len(batch), 11, 64),
            'context_features': torch.randn(len(batch), 8),
            'attention_mask': torch.ones(len(batch), 11),
        },
        'n_events': torch.tensor([b['n_events'] for b in batch]),
    }

print("Dataset classes defined.")

In [ ]:
# Create datasets
print("Creating datasets...")
train_dataset = SyntheticPlayerDataset(n_players=100, n_samples_per_player=5)
val_dataset = SyntheticPlayerDataset(n_players=100, n_samples_per_player=2)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn,
)

print(f"✓ Train samples: {len(train_dataset)}")
print(f"✓ Val samples: {len(val_dataset)}")

## 3. Model Setup

In [ ]:
print("Initializing model components...")

# Event Graph Encoder
event_encoder = EventGraphEncoder(
    node_input_dim=64,
    node_embedding_dim=64,
    context_dim=8,
    n_gat_layers=2,
    n_attention_heads=4,
    dropout=0.1,
)

# Temporal Transformer
temporal_encoder = TemporalTransformerEncoder(
    input_dim=64,
    hidden_dim=128,
    n_layers=2,
    n_heads=4,
    max_seq_length=100,
    dropout=0.1,
)

# Fusion Head
fusion_head = FusionHead(
    data_dim=128,
    global_dim=128,
    hidden_dim=128,
    output_dim=128,
    n_roles=10,
)

print("✓ Event Encoder initialized")
print("✓ Temporal Transformer initialized")
print("✓ Fusion Head initialized")

In [ ]:
# Simple combined model for demonstration
class SimplePlayerModel(nn.Module):
    """Simplified model for demonstration."""
    
    def __init__(self, temporal_encoder, fusion_head):
        super().__init__()
        self.temporal_encoder = temporal_encoder
        self.fusion_head = fusion_head
    
    def forward(self, event_batch, sequence_batch, **kwargs):
        # Skip event encoder for demo, use pre-computed embeddings
        temporal_output = self.temporal_encoder(
            event_embeddings=sequence_batch['event_embeddings'],
            time_positions=sequence_batch['time_positions'],
            attention_mask=sequence_batch['attention_mask'],
        )
        
        fusion_output = self.fusion_head(
            data_embedding=temporal_output['player_embedding'],
            global_embedding=None,
            external_features=None,
        )
        
        return {
            'embedding': fusion_output['embedding'],
            'reliability': fusion_output['reliability'],
            'role_logits': fusion_output['role_logits'],
            'data_embedding': fusion_output['data_embedding'],
        }

model = SimplePlayerModel(temporal_encoder, fusion_head)
n_params = sum(p.numel() for p in model.parameters())
print(f"\n✓ Combined model created")
print(f"  Total parameters: {n_params:,}")

## 4. Training

In [ ]:
print("=" * 60)
print("TRAINING")
print("=" * 60)

# Training config
training_config = TrainingConfig(
    learning_rate=1e-4,
    batch_size=32,
    max_epochs=10,
    warmup_epochs=2,
    patience=5,
    device=str(device),
    checkpoint_dir='checkpoints_demo',
)

# Create trainer
trainer = Trainer(
    model=model,
    config=training_config,
    train_loader=train_loader,
    val_loader=val_loader,
)

print(f"\nTraining Configuration:")
print(f"  Learning rate: {training_config.learning_rate}")
print(f"  Batch size: {training_config.batch_size}")
print(f"  Max epochs: {training_config.max_epochs}")
print(f"  Device: {device}")

In [ ]:
# Train
print("\nStarting training...")
history = trainer.train(n_epochs=10)

## 5. Training Results

In [ ]:
print("=" * 60)
print("TRAINING RESULTS")
print("=" * 60)

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
if 'total_loss' in history:
    axes[0].plot(history['total_loss'], label='Train Loss', marker='o')
if 'val_total_loss' in history:
    axes[0].plot(history['val_total_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training Loss', fontsize=14)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Learning rate
if 'lr' in history:
    axes[1].plot(history['lr'], color='green', marker='o')
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Learning Rate', fontsize=12)
    axes[1].set_title('Learning Rate Schedule', fontsize=14)
    axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Export Embeddings

In [ ]:
print("Exporting embeddings...")

# Get embeddings for all players
model.eval()
all_embeddings = []
all_player_ids = []

with torch.no_grad():
    for batch in val_loader:
        # Move to device
        sequences = {k: v.to(device) for k, v in batch['sequences'].items()}
        events = {k: v.to(device) for k, v in batch['events'].items()}
        
        outputs = model(
            event_batch=events,
            sequence_batch=sequences,
        )
        
        all_embeddings.append(outputs['embedding'].cpu())
        all_player_ids.append(batch['player_ids'])

all_embeddings = torch.cat(all_embeddings, dim=0)
all_player_ids = torch.cat(all_player_ids, dim=0)

print(f"✓ Exported {len(all_embeddings)} embeddings")
print(f"  Embedding shape: {all_embeddings.shape}")

In [ ]:
# Save embeddings
embeddings_dict = {
    pid.item(): emb.numpy() 
    for pid, emb in zip(all_player_ids, all_embeddings)
}

np.savez('player_embeddings.npz', **{str(k): v for k, v in embeddings_dict.items()})
print("✓ Saved: player_embeddings.npz")

print("\n" + "=" * 60)
print("✅ TRAINING COMPLETE!")
print("=" * 60)